In [ ]:
"""
Futures Execution Algorithm - File Dialog Version
Double-click to run, then select your data file
"""

import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime
from futures_execution import FuturesExecutionAlgo  # Your main code file

# Try to import tkinter for file dialog (comes with Python)
try:
    import tkinter as tk
    from tkinter import filedialog, messagebox
    HAS_TKINTER = True
except ImportError:
    HAS_TKINTER = False
    print("Warning: tkinter not available. Using command line input instead.")

def select_file_gui():
    """Open a file dialog for the user to select a file"""
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    root.attributes('-topmost', True)  # Bring dialog to front
    
    file_path = filedialog.askopenfilename(
        title="Select Training Data File",
        filetypes=[
            ("CSV files", "*.csv"),
            ("Excel files", "*.xlsx *.xls"),
            ("All files", "*.*")
        ]
    )
    
    root.destroy()
    return file_path

def select_file_cli():
    """Command line fallback if GUI not available"""
    print("\nEnter the path to your data file:")
    print("(You can drag and drop the file into this window)")
    file_path = input("File path: ").strip().strip('"').strip("'")
    return file_path

def validate_file(file_path):
    """Check if file exists and has required columns"""
    if not os.path.exists(file_path):
        return False, f"File not found: {file_path}"
    
    # Try to read first few rows to check columns
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path, nrows=5)
        elif file_path.endswith(('.xlsx', '.xls')):
            df = pd.read_excel(file_path, nrows=5)
        else:
            return False, "Unsupported file format. Please use CSV or Excel."
        
        # Check required columns
        required_cols = ['close', 'bid', 'ask', 'bid_volume', 'ask_volume', 'volume']
        missing = [col for col in required_cols if col not in df.columns]
        
        if missing:
            return False, f"Missing required columns: {missing}"
        
        return True, "File is valid"
        
    except Exception as e:
        return False, f"Error reading file: {str(e)}"

def load_data(file_path):
    """Load data from file"""
    if file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format")

def main():
    print("=" * 60)
    print("FUTURES EXECUTION ALGORITHM")
    print("=" * 60)
    print("\nThis program will train and test a futures execution algorithm")
    print("using your market data.\n")
    
    # Step 1: Select file
    print("STEP 1: SELECT YOUR DATA FILE")
    print("-" * 40)
    
    file_path = None
    
    if HAS_TKINTER:
        print("Opening file dialog...")
        file_path = select_file_gui()
    else:
        file_path = select_file_cli()
    
    if not file_path:
        print("\nNo file selected. Exiting...")
        input("\nPress Enter to exit...")
        sys.exit(0)
    
    # Validate file
    print(f"\nSelected file: {os.path.basename(file_path)}")
    print("Validating file...")
    
    is_valid, message = validate_file(file_path)
    if not is_valid:
        print(f"\nERROR: {message}")
        input("\nPress Enter to exit...")
        sys.exit(1)
    
    print("✓ File is valid!")
    
    # Step 2: Load data
    print("\nSTEP 2: LOADING DATA")
    print("-" * 40)
    
    try:
        data = load_data(file_path)
        print(f"Loaded {len(data)} rows of data")
        print(f"Date range: {data.index[0] if isinstance(data.index, pd.DatetimeIndex) else 'N/A'}")
    except Exception as e:
        print(f"ERROR loading data: {str(e)}")
        input("\nPress Enter to exit...")
        sys.exit(1)
    
    # Step 3: Split into training and testing
    print("\nSTEP 3: SPLITTING DATA")
    print("-" * 40)
    
    # Use 80% for training, 20% for testing
    split_point = int(len(data) * 0.8)
    train_data = data.iloc[:split_point].copy()
    test_data = data.iloc[split_point:].copy()
    
    print(f"Training data: {len(train_data)} rows ({split_point/len(data)*100:.1f}%)")
    print(f"Testing data: {len(test_data)} rows ({100-split_point/len(data)*100:.1f}%)")
    
    # Step 4: Configure training
    print("\nSTEP 4: CONFIGURATION")
    print("-" * 40)
    
    try:
        episodes = int(input("Number of training episodes (default=100): ") or "100")
        order_size = int(input("Order size (default=1000): ") or "1000")
    except ValueError:
        print("Invalid input. Using defaults.")
        episodes = 100
        order_size = 1000
    
    # Step 5: Initialize and train
    print("\nSTEP 5: INITIALIZING ALGORITHM")
    print("-" * 40)
    
    algo = FuturesExecutionAlgo(symbol="ES")
    
    print("\nSTEP 6: TRAINING")
    print("-" * 40)
    print(f"Training for {episodes} episodes...")
    print("(This may take a few minutes)\n")
    
    try:
        scores, avg_scores = algo.train(train_data, episodes=episodes)
        print("\n✓ Training completed!")
    except Exception as e:
        print(f"\nERROR during training: {str(e)}")
        input("\nPress Enter to exit...")
        sys.exit(1)
    
    # Step 7: Testing
    print("\nSTEP 7: TESTING")
    print("-" * 40)
    
    try:
        result = algo.execute_order(test_data, order_size=order_size)
        print("✓ Testing completed!")
    except Exception as e:
        print(f"\nERROR during testing: {str(e)}")
        input("\nPress Enter to exit...")
        sys.exit(1)
    
    # Step 8: Results
    print("\n" + "=" * 60)
    print("EXECUTION RESULTS")
    print("=" * 60)
    print(f"\nData file: {os.path.basename(file_path)}")
    print(f"Training episodes: {episodes}")
    print(f"Order size: {order_size}")
    print("\n" + "-" * 40)
    print(f"Executed: {result['executed_quantity']:,}/{result['total_quantity']:,}")
    print(f"Completion rate: {result['completion_rate']*100:.2f}%")
    print(f"Average execution price: ${result['average_execution_price']:.2f}")
    print(f"Arrival price: ${result['arrival_price']:.2f}")
    print(f"Implementation shortfall: ${result['implementation_shortfall']:.2f}")
    
    # Step 9: Save results
    print("\nSTEP 9: SAVING RESULTS")
    print("-" * 40)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    result_file = f"execution_results_{timestamp}.txt"
    
    with open(result_file, "w") as f:
        f.write("FUTURES EXECUTION RESULTS\n")
        f.write("=" * 40 + "\n\n")
        f.write(f"Data file: {os.path.basename(file_path)}\n")
        f.write(f"Training episodes: {episodes}\n")
        f.write(f"Order size: {order_size}\n")
        f.write(f"Training data rows: {len(train_data)}\n")
        f.write(f"Testing data rows: {len(test_data)}\n\n")
        f.write("-" * 40 + "\n")
        f.write(f"Executed: {result['executed_quantity']}/{result['total_quantity']}\n")
        f.write(f"Completion rate: {result['completion_rate']*100:.2f}%\n")
        f.write(f"Average price: ${result['average_execution_price']:.2f}\n")
        f.write(f"Arrival price: ${result['arrival_price']:.2f}\n")
        f.write(f"Implementation shortfall: ${result['implementation_shortfall']:.2f}\n")
    
    print(f"Results saved to: {result_file}")
    
    print("\n" + "=" * 60)
    print("PROGRAM COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    
    input("\nPress Enter to exit...")

if __name__ == "__main__":
    main()